# **Instalações necessárias**

In [ ]:
%pip install opencv-python

In [ ]:
# Instalação das dependências do Pipeline de Polarimetria
%pip install "numpy<2" scipy pandas matplotlib astropy photutils astropop

In [ ]:
%pip install --force-reinstall --no-deps matplotlib

In [ ]:
%pip install --force-reinstall matplotlib kiwisolver pillow cycler pyparsing contourpy

In [ ]:
%pip install --force-reinstall pillow

# **Bibliotecas**

In [ ]:
import os
import glob
import warnings
import concurrent.futures
import numpy as np
import matplotlib.pyplot as plt
import glob
import pandas as pd
import shutil
import cv2

In [ ]:
from astropy.io import fits
from astropy import units as u
from astropy.utils.exceptions import AstropyWarning
from astropy.stats import sigma_clipped_stats, sigma_clip

In [ ]:
from astropop.image import imcombine, processing, imarith
from astropop.framedata import read_framedata
from astropop.file_collection import FitsFileGroup
from astropop.polarimetry import SLSDualBeamPolarimetry

In [ ]:
from photutils.detection import DAOStarFinder
from photutils.centroids import centroid_com
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry

In [ ]:
from collections import defaultdict
from matplotlib.patches import Circle

In [ ]:
# Silencia os avisos do HIERARCH
warnings.simplefilter('ignore', category=AstropyWarning)

# **Caminho dos Dados do Usuário**

In [ ]:
#Caminho até chegar chegar antes na pasta da noite
CAMINHO_PASTA__REDUCAO = '/home/likeableng/BeACoN/arquivos_de_reducao/'

In [ ]:
#Colocar os dados da noite
DATA = "20180411" #ex: "AAAAMMDD"
NOME_ESTRELA = "hd148937" #ex: "paqr" "osco"

# **Ajuste Pré redução**

In [ ]:
#Ajuste data
mes = int(DATA[4:6])

nome_mes = ['', 'jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 
            'ago', 'set', 'out', 'nov', 'dez']

data_ajustada = DATA[2:4]+nome_mes[mes]+DATA[6:]

In [ ]:
#Caminhos para pastas reduced e raw

raw_dir = f"{CAMINHO_PASTA__REDUCAO}{data_ajustada}/raw/"

reduced_dir = f"{CAMINHO_PASTA__REDUCAO}{data_ajustada}/reduced/"
os.makedirs(reduced_dir, exist_ok=True)

In [ ]:
#Caminho para dados estrela
pasta_raw_estrela = f'{raw_dir}{NOME_ESTRELA}'
pasta_reduced_estrela = f'{reduced_dir}{NOME_ESTRELA}'
os.makedirs(pasta_reduced_estrela, exist_ok=True)

#Caminho para dados calib
pasta_raw_calib = f'{raw_dir}calib/'
pasta_reduced_calib = f'{reduced_dir}calib/'
os.makedirs(pasta_reduced_calib, exist_ok=True)

print(pasta_reduced_estrela)

In [ ]:
# Cria a pasta para o filtro e move o ficheiro para a respetiva pasta de filtro

# Percorre todos os ficheiros na pasta da estrela
for nome_ficheiro in os.listdir(pasta_raw_estrela):
    caminho_origem = os.path.join(pasta_raw_estrela, nome_ficheiro)

    if os.path.isfile(caminho_origem) and nome_ficheiro.endswith('.fits'):
        partes = nome_ficheiro.split('_')

        # Se o arquivo tiver 'g5' no nome (ex: hd148937_g5_v_031.fits)
        if 'g5' in nome_ficheiro.lower() and len(partes) >= 4:
            etiqueta_filtro = partes[2]  # Pega o filtro no índice 2 ('v', 'b', etc.)
        # Para nomes padrão como '48lib_F1_001.fits'
        elif len(partes) >= 3:
            etiqueta_filtro = partes[1]  # Pega o filtro no índice 1 ('F1', etc.)
        else:
            continue

        pasta_filtro = os.path.join(pasta_raw_estrela, etiqueta_filtro)
        os.makedirs(pasta_filtro, exist_ok=True)

        caminho_destino = os.path.join(pasta_filtro, nome_ficheiro)
        shutil.move(caminho_origem, caminho_destino)
        # print(f"Movido: {nome_ficheiro} -> {etiqueta_filtro}/")

print("\n✅ Todos os ficheiros foram organizados nas respetivas pastas de filtro!")

In [ ]:
#Função para extrair o número final do arquivo para ordenação correta
for item in os.listdir(pasta_raw_estrela):
    caminho_pasta_filtro = os.path.join(pasta_raw_estrela, item)
    
    # Processa apenas se for um diretório (pasta de filtro)
    if os.path.isdir(caminho_pasta_filtro):
        arquivos_fits = glob.glob(os.path.join(caminho_pasta_filtro, '*.fits'))
        
        if not arquivos_fits:
            continue
            
        
        def extrair_numero(caminho):
            nome = os.path.basename(caminho)
            parte_num = nome.replace('.fits', '').split('_')[-1]
            return int(parte_num) if parte_num.isdigit() else 0

        arquivos_ordenados = sorted(arquivos_fits, key=extrair_numero)

        #print(f"\n--- Renumerando arquivos em {item} ({len(arquivos_ordenados)} imagens) ---")

        # Renomeia mantendo o prefixo e redefinindo a contagem para 001, 002, 003...
        for idx, caminho_antigo in enumerate(arquivos_ordenados, start=1):

            nome_antigo = os.path.basename(caminho_antigo)
            partes = nome_antigo.replace('.fits', '').split('_')
            
            prefixo = "_".join(partes[:-1])
            
            novo_nome = f"{prefixo}_{idx:03d}.fits"
            caminho_novo = os.path.join(caminho_pasta_filtro, novo_nome)
            
            os.rename(caminho_antigo, caminho_novo)
            #print(f"{nome_antigo}  ->  {novo_nome}")


In [ ]:
#Move cada arquivo de dentro da subpasta para a pasta_raw_estrela e apaga as subpastas 
for item in os.listdir(pasta_raw_estrela):
    caminho_subpasta = os.path.join(pasta_raw_estrela, item)
    
    if os.path.isdir(caminho_subpasta):
        print(f"Movendo arquivos da pasta '{item}' de volta...")
        
        for nome_arquivo in os.listdir(caminho_subpasta):
            caminho_origem = os.path.join(caminho_subpasta, nome_arquivo)
            caminho_destino = os.path.join(pasta_raw_estrela, nome_arquivo)
            
            if os.path.isfile(caminho_origem):
                shutil.move(caminho_origem, caminho_destino)
        
        os.rmdir(caminho_subpasta)
       

# **Funções Motoras**

In [ ]:
#Ajustar nome dos arquivos, alterando o filtro de F# para b,v,r,i

mapeamento = {
    'F1': 'b',
    'F2': 'v',
    'F3': 'r',
    'F4': 'i'
}

def ajustar_nome_filtro(caminho_da_pasta):
# Percorre todos os arquivos dentro da pasta especificada
    for nome_arquivo in os.listdir(caminho_da_pasta):
        if nome_arquivo.endswith('.fits'):
            for chave, letra in mapeamento.items():
                marcador_antigo = f'_{chave}_'
                marcador_novo = f'_{letra}_'
            
                if marcador_antigo in nome_arquivo:
                    # É preciso juntar o caminho da pasta com o nome do arquivo
                    caminho_antigo = os.path.join(caminho_da_pasta, nome_arquivo)
                    novo_nome = nome_arquivo.replace(marcador_antigo, marcador_novo)
                    caminho_novo = os.path.join(caminho_da_pasta, novo_nome)
                
                    os.rename(caminho_antigo, caminho_novo)
                    print(f'{nome_arquivo}  ->  {novo_nome}')
                    break


In [ ]:
#Transformar imagens 3D em 2D

def descompactar_cubos_3d(pasta):    
    # Achar arquivos FITS na pasta de entrada
    arquivos_fits = glob.glob(os.path.join(pasta, '*.fits'))
    print(f"Encontrados {len(arquivos_fits)} arquivos na pasta de entrada.\n")
    
    for caminho_arquivo in arquivos_fits:
        nome_base = os.path.basename(caminho_arquivo).replace('.fits', '')

        try:
            # Ignora o ficheiro se estiver vazio (0 bytes) para evitar OSError
            if os.path.getsize(caminho_arquivo) == 0:
                print(f"Ignorando '{nome_base}.fits' (Ficheiro vazio/0 bytes).")
                continue

            dados, header = fits.getdata(caminho_arquivo, header=True)
            
            # Verifica a dimensão da matriz
            if dados.ndim == 3:
                num_imagens = dados.shape[0]
                print(f"Descompactando '{nome_base}.fits' ({num_imagens} fatias)...")
                
                for i in range(num_imagens):
                    fatia_2d = dados[i]
                    
                    # Cria um nome sequencial (ex: bias_001_001.fits)
                    nome_saida = f"{nome_base}_{i+1:03d}.fits"
                    caminho_saida = os.path.join(pasta, nome_saida)
                    
                    # Salva a fatia 2D com o cabeçalho original.
                    fits.writeto(caminho_saida, fatia_2d, header, overwrite=True)
                    
            elif dados.ndim == 2:
                print(f"Ignorando '{nome_base}.fits' (Já é 2D).")
            else:
                print(f"Ignorando '{nome_base}.fits' (Formato não suportado).")
                
        except OSError as e:
            # Captura especificamente ficheiros corrompidos ou não suportados pelo astropy
            print(f"Erro ao ler '{nome_base}.fits': Ficheiro corrompido ou inválido. A ignorar...")
        except Exception as e:
            # Previne que outros erros parem o loop
            print(f"Erro inesperado no ficheiro '{nome_base}.fits': {e}")
            
    print("\nDescompactação concluída!")


In [ ]:
# ============================================================
# Grupos de calibração identificados pelo NOME do arquivo
# ------------------------------------------------------------
# Aceita, por exemplo:
#   bias_004            bias_g5_005        bias_full_001        bias_full_g5_001
#   flat_F4_004         flat_g5_F2_001     flat_full_F1_001     flat_full_g5_F1_001
#
# Regra de pareamento: cada grupo só usa calibração do MESMO grupo
#   (full com full, g5 com g5, full_g5 com full_g5, sem tag com sem tag).
# Para aceitar um novo tipo no futuro, basta adicioná-lo em TAGS_CALIB.
# ============================================================
TAGS_CALIB = ('full', 'g5')

def extrair_tag_calib(nome_arquivo, preamp_val=None):
    """
    Devolve o grupo do arquivo: '', 'g5', 'full' ou 'full_g5'.
    Primeiro olha o nome do arquivo. Se não houver nenhuma tag no nome,
    mantém o comportamento antigo (PREAMP do cabeçalho: diferente de '1' vira 'g<PREAMP>').
    """
    partes = os.path.basename(nome_arquivo).lower().replace('.fits', '').split('_')
    tags = [t for t in TAGS_CALIB if t in partes]   # ordem fixa: full antes de g5
    if tags:
        return "_".join(tags)
    if preamp_val is not None and preamp_val != '1':
        return f"g{preamp_val}"
    return ""

def sufixo_calib(tag):
    """'' -> ''   |   'full' -> '_full'   |   'full_g5' -> '_full_g5'"""
    return f"_{tag}" if tag else ""

def limpar_preamp(hdr):
    # Ex.: '5.1x' vira '5'
    return str(hdr.get('PREAMP', 'UNKNOWN')).lower().replace('x', '').split('.')[0]

# ------------------------------------------------------------
# Imagens sem tag no nome: detectar o modo "full" pelo cabeçalho
# SUBRECT = '(x0), 1024, 1024, (y0)'  -> largura e altura nos valores do meio
# ------------------------------------------------------------
TAG_SUBRECT_1024 = 'full'   # grupo de calibração usado para imagens 1024x1024

def eh_subrect_1024(hdr):
    try:
        valores = [int(float(v)) for v in str(hdr.get('SUBRECT', '')).replace('(', '').replace(')', '').replace("'", '').split(',')]
        return len(valores) >= 4 and valores[1] == 1024 and valores[2] == 1024
    except (ValueError, TypeError):
        return False

def tag_calib_imagem(caminho, hdr):
    """
    Grupo de calibração de uma imagem da estrela:
    tag do nome (ou PREAMP, regra antiga) + TAG_SUBRECT_1024 se o SUBRECT for 1024x1024.
    Ex.: sem tag + 1024x1024 -> 'full' | g5 + 1024x1024 -> 'full_g5'
    """
    tag = extrair_tag_calib(caminho, limpar_preamp(hdr))
    if eh_subrect_1024(hdr):
        partes = [p for p in tag.split('_') if p] + [TAG_SUBRECT_1024]
        ordem = list(TAGS_CALIB)
        partes = sorted(set(partes), key=lambda p: ordem.index(p) if p in ordem else len(ordem))
        tag = "_".join(partes)
    return tag


In [ ]:
#Fazer o Master BIAS - sem usar a extração de raios cósmicos
# Um master por grupo de nome: sem tag, g5, full, full_g5 (ver extrair_tag_calib)

def criar_master_bias(pasta_entrada, pasta_saida):
    print("\n[Calibração] Criando Master Bias por grupo (sem tag / g5 / full / full_g5)...")
    fg = FitsFileGroup(location=pasta_entrada, fits_ext=['.fits'], ext=0)
    
    bias_files = fg.filtered({'OBJECT': 'bias'}).files
    
    arquivos_por_grupo = defaultdict(list)
    for arq in bias_files:
        hdr = fits.getheader(arq)
        if hdr.get('NAXIS') == 2:
            preamp_val = limpar_preamp(hdr)
            tag = extrair_tag_calib(arq, preamp_val)
            arquivos_por_grupo[tag].append(arq)
            
    for tag, arq_2d in arquivos_por_grupo.items():
        print(f"-> Processando Bias [{tag or 'padrão'}] ({len(arq_2d)} arquivos)")
        
        frames = [read_framedata(f, unit='adu', use_memmap_backend=True) for f in arq_2d]
        gain = float(frames[0].header['GAIN']) * u.electron/u.adu
        
        for frame in frames:
            processing.gain_correct(frame, gain, inplace=True)
            # Sem L.A. Cosmic para processamento rápido
            
        master = imcombine(frames, method='median')
        master.meta['astropop master_type'] = 'bias'
        master.meta['PREAMP'] = limpar_preamp(frames[0].header)
        master.meta['CALTAG'] = tag if tag else 'padrao'
        
        caminho_out = os.path.join(pasta_saida, f'master_bias{sufixo_calib(tag)}.fits')
        
        master.write(caminho_out, overwrite=True, no_fits_standard_units=True)
        print(f"   Salvo: {caminho_out}\n")
        del frames


In [ ]:
#Fazer o Mater Flat - sem usar a extração de raios cósmicos

letra_filtro = ["", "b", "v", "r", "i"]

def criar_master_flat(pasta_in, pasta_out):
    main_fg = FitsFileGroup(location=pasta_in, fits_ext=['.fits'], ext=0)
    flat_files = main_fg.filtered({'OBJECT': 'flat'}).files

    # 1. Agrupamento manual duplo (por Filtro e grupo de nome: sem tag / g5 / full / full_g5)
    arquivos_por_grupo = defaultdict(list)
    for arq in flat_files:
        hdr = fits.getheader(arq)
        if hdr.get('NAXIS') == 2:
            preamp_val = limpar_preamp(hdr)
            tag = extrair_tag_calib(arq, preamp_val)
            
            # Extrai o filtro
            filtro_val = str(hdr.get('FILTER', 'UNKNOWN'))
            
            # Agrupa os dois juntos
            arquivos_por_grupo[(filtro_val, tag)].append(arq)

    # 2. Processa cada grupo separadamente
    for (filtro_val, tag), arquivos_flat_2d in arquivos_por_grupo.items():
        print(f"\n--- Processando Master Flat | Filtro: {filtro_val} | Grupo: {tag or 'padrão'} ({len(arquivos_flat_2d)} arquivos) ---")
        
        # Carrega SOMENTE o bias do mesmo grupo (full só com full, full_g5 só com full_g5, ...)
        sufixo = sufixo_calib(tag)
        caminho_master_bias = os.path.join(pasta_out, f'master_bias{sufixo}.fits')
        
        if not os.path.exists(caminho_master_bias):
            print(f"ERRO: {caminho_master_bias} não encontrado! Abortando este flat.")
            continue
            
        master_bias = read_framedata(caminho_master_bias)

        flat_frames = [read_framedata(f, unit='adu', use_memmap_backend=True) for f in arquivos_flat_2d]
        gain = float(flat_frames[0].header['GAIN']) * u.electron / u.adu

        for f_idx, frame in enumerate(flat_frames):
            print(f" -> Processando flat [{filtro_val}] {f_idx+1}/{len(flat_frames)}", end='\r')
            processing.gain_correct(frame, gain, inplace=True)
            processing.subtract_bias(frame, master_bias, inplace=True)

        print(f"\nCombinando e normalizando Master Flat ({filtro_val}, grupo {tag or 'padrão'})...")
        master_flat = imcombine(flat_frames, method='median')
        mean_value = master_flat.mean()
        master_flat = imarith(master_flat, mean_value, '/')

        master_flat.meta['astropop master_type'] = 'flat'
        master_flat.meta['FILTER'] = filtro_val
        master_flat.meta['PREAMP'] = limpar_preamp(flat_frames[0].header)
        master_flat.meta['CALTAG'] = tag if tag else 'padrao'

        valor_filtro = filtro_val[1:]
        numero_filtro = int(valor_filtro)
        letra = letra_filtro[numero_filtro]

        # Salva com o nome correto (ex.: master_flat_full_g5_b.fits)
        nome_out = f'master_flat{sufixo}_{letra}.fits'
        
        mflat_path = os.path.join(pasta_out, nome_out)
        master_flat.write(mflat_path, overwrite=True, no_fits_standard_units=True)

        print(f"Salvo: {mflat_path}")
        del flat_frames


In [ ]:
#Função auxiliar da redução de imagens em paralelo
def dividir_em_lotes(lista, tamanho):
    return [lista[i:i + tamanho] for i in range(0, len(lista), tamanho)]

def processar_lote_por_filtro(filtro, preamp, arquivos, caminho_mbias, caminho_mflat, pasta_saida):
    master_bias = read_framedata(caminho_mbias)
    master_flat = read_framedata(caminho_mflat)
    
    total = len(arquivos)
    for i, arq in enumerate(arquivos):
        nome_arq = os.path.basename(arq)
        
        frame = read_framedata(arq, unit='adu', use_memmap_backend=True)
        gain = float(frame.header['GAIN']) * u.electron / u.adu

        processing.cosmics_lacosmic(frame, inplace=True)
        processing.gain_correct(frame, gain, inplace=True)
        processing.subtract_bias(frame, master_bias, inplace=True)
        processing.flat_correct(frame, master_flat, inplace=True)

        frame.meta['REDUCED'] = True
        
        # AQUI ADICIONAMOS O PREFIXO "red_"
        caminho_saida = os.path.join(pasta_saida, f"red_{nome_arq}")
        
        frame.write(caminho_saida, overwrite=True, no_fits_standard_units=True)
        
    return f"Lote [Filtro {filtro} | grupo {preamp}] ({total} imagens) CONCLUÍDO!"

In [ ]:
#Para que não vá em uma sequência linear

def reduzir_objetos_paralelo(pasta_raw_obj, pasta_red_calib, pasta_red_obj, tag_forcada=None):
    os.makedirs(pasta_red_obj, exist_ok=True)

    arquivos_fits = glob.glob(os.path.join(pasta_raw_obj, '*.fits'))
    
    if not arquivos_fits:
        print(f"[AVISO] Nenhum arquivo FITS encontrado na pasta: {pasta_raw_obj}")
        return

    # Agrupamento seguro
    arquivos_por_grupo = defaultdict(list)
    for arq in arquivos_fits:
        try:
            hdr = fits.getheader(arq)
            if hdr.get('NAXIS') == 2:
                tag = tag_forcada if tag_forcada is not None else tag_calib_imagem(arq, hdr)
                
                filtro_val = str(hdr.get('FILTER', 'UNKNOWN'))
                arquivos_por_grupo[(filtro_val, tag)].append(arq)
        except Exception as e:
            print(f"Erro ao ler cabeçalho de {os.path.basename(arq)}: {e}")

    print("Iniciando processamento paralelo em pequenos lotes seguros...")

    with concurrent.futures.ProcessPoolExecutor() as executor:
        futuros = []
        
        for (filtro_val, tag), arquivos_2d in arquivos_por_grupo.items():
            if not arquivos_2d:
                continue
            
            sufixo = sufixo_calib(tag)

            valor_filtro = filtro_val[1:]
            numero_filtro = int(valor_filtro)
            letra = letra_filtro[numero_filtro]
            
            caminho_mbias = os.path.join(pasta_red_calib, f'master_bias{sufixo}.fits')
            caminho_mflat = os.path.join(pasta_red_calib, f'master_flat{sufixo}_{letra}.fits')
            
            if not os.path.exists(caminho_mbias) or not os.path.exists(caminho_mflat):
                print(f"[AVISO] Calibração ausente para Filtro {filtro_val} ({letra}), grupo '{tag or 'padrão'}'. Pulando conjunto.")
                continue
            
            lotes = dividir_em_lotes(arquivos_2d, 50)
            
            for lote in lotes:
                futuro = executor.submit(processar_lote_por_filtro, filtro_val, tag or 'padrão', lote, caminho_mbias, caminho_mflat, pasta_red_obj)
                futuros.append(futuro)

        for futuro in concurrent.futures.as_completed(futuros):
            print(futuro.result())

    print("\n\nRedução de TODOS os objetos concluída com sucesso!")


In [ ]:
#Rastrear os centros para par
def rastrear_centroide_ancorado(dados, x_ancora, y_ancora, janela=12):

    x_int, y_int = int(round(x_ancora)), int(round(y_ancora))
    h, w = dados.shape
    
    y_min, y_max = max(0, y_int - janela), min(h, y_int + janela + 1)
    x_min, x_max = max(0, x_int - janela), min(w, x_int + janela + 1)
    
    recorte = dados[y_min:y_max, x_min:x_max]
    recorte = np.nan_to_num(recorte, nan=np.nanmedian(recorte))
    
    # Subtrai mediana local para o centróide focar no pico da estrela
    recorte_sub = np.maximum(recorte - np.median(recorte), 0)
    xc_rel, yc_rel = centroid_com(recorte_sub)
    
    x_novo = x_min + xc_rel
    y_novo = y_min + yc_rel
    
    # Se o desvio for fora do limite aceitável, segura na posição de âncora
    if np.abs(x_novo - x_ancora) > (janela - 2) or np.abs(y_novo - y_ancora) > (janela - 2):
        return x_ancora, y_ancora
        
    return x_novo, y_novo

# **Calibração**

In [ ]:
#Ajustar nome F# na pasta calib
ajustar_nome_filtro(pasta_raw_calib)

In [ ]:
#Ajustar nome F# na pasta da estrela
ajustar_nome_filtro(pasta_raw_estrela)

# **Ajuste de Quantidade de Observações do Objeto na Noite**

In [ ]:
# Achar arquivos para verificar a quantidade de lâminas (compatível com 'g5' e subpastas)

filtros_testar = ['b', 'v', 'r', 'i', 'u', 'f1', 'f2']
arquivos_encontrados = []

for filtro in filtros_testar:
    # Padrão flexível: aceita 'Estrela_b_*.fits' e 'Estrela_g5_b_*.fits' sem exigir zero inicial
    padrao_busca = os.path.join(pasta_raw_estrela, f"{NOME_ESTRELA}*_{filtro}_*.fits")
    arquivos_encontrados = sorted(glob.glob(padrao_busca))

    
    if not arquivos_encontrados:
        padrao_subpastas = os.path.join(pasta_raw_estrela, "**", f"{NOME_ESTRELA}*_{filtro}_*.fits")
        arquivos_encontrados = sorted(glob.glob(padrao_subpastas, recursive=True))

    if arquivos_encontrados:
        print(f" Arquivos identificados para análise no filtro '{filtro.upper()}': {len(arquivos_encontrados)} arquivos.")
        break


if not arquivos_encontrados:
    padrao_generico = os.path.join(pasta_raw_estrela, "**", "*.fits")
    arquivos_encontrados = sorted(glob.glob(padrao_generico, recursive=True))
    if arquivos_encontrados:
        print(f"⚠️ Padrão específico não encontrado. Usando todos os {len(arquivos_encontrados)} arquivos .fits localizados.")

In [ ]:
#Verificando as lâminas
valores_lamina = []

for arq in arquivos_encontrados:
    try:
        hdr = fits.getheader(arq)

        # O valor -999.0 serve como aviso caso a palavra-chave não exista
        valor = hdr.get('LAMINA', -999.0)

        valor = valor[1:]

        if valor != -999.0:
            valores_lamina.append(valor)
            
    except Exception as e:
        print(f"Erro ao ler o ficheiro {os.path.basename(arq)}: {e}")

if valores_lamina:
    valor_mais_alto = max(valores_lamina)
    vezes_repetido = valores_lamina.count(valor_mais_alto)
    
    print("=== RESULTADOS DA LÂMINA ===")
    print(f"Total de leituras bem-sucedidas: {len(valores_lamina)}")
    print(f"Valor mais alto encontrado: {valor_mais_alto}")
    print(f"O valor mais alto aparece {vezes_repetido} vez(es) nas imagens.")

else:
    pastas = [
        f
        for f in os.listdir(pasta_raw_estrela)
        if os.path.isdir(os.path.join(pasta_raw_estrela, f))
    ]
    vezes_repetido = len(pastas)
    #print(vezes_repetido)

    padrao_busca = os.path.join(f"{pasta_raw_estrela}//Obs_1_{NOME_ESTRELA}", f'{NOME_ESTRELA}_b_0*.fits')
    arquivos_encontrados_i = sorted(glob.glob(padrao_busca))
    for arq in arquivos_encontrados_i:
        try:
            hdr = fits.getheader(arq)

            # O valor -999.0 serve como aviso caso a palavra-chave não exista
            valor = hdr.get('LAMINA', -999.0)

            valor = valor[1:]

            if valor != -999.0:
                valores_lamina.append(valor)             
        except Exception as e:
            print(f"Erro ao ler o ficheiro {os.path.basename(arq)}: {e}")

        valor_mais_alto = max(valores_lamina)
    print(valores_lamina)

    

In [ ]:
#Definindo a quantidade de lâminas e portanto a quantidade de imagens 
# por pasta de cada filtro de acordo com o nome da lâmina
if valor_mais_alto == "B":
    numero_de_arquivos_por_pasta = 8

else:
    numero_de_arquivos_por_pasta = 16
    

In [ ]:
#Criando as pastas de cada observação da estrela

i=0 
while i < vezes_repetido:
    i=i+1 
    os.makedirs(f"{pasta_raw_estrela}/Obs_{i}_{NOME_ESTRELA}", exist_ok=True)

    os.makedirs(f"{pasta_reduced_estrela}/Obs_{i}_{NOME_ESTRELA}", exist_ok=True)

In [ ]:
#Passando os arquivos de acordo com a numeração da observação (compatível com 'g5')

for nome_item in os.listdir(pasta_raw_estrela):
    caminho_origem_completo = os.path.join(pasta_raw_estrela, nome_item)

    if os.path.isfile(caminho_origem_completo) and nome_item.endswith('.fits'):
        partes = nome_item.split('_')

        if len(partes) >= 3:
            final_do_item = partes[-1]
            valor_do_item = os.path.splitext(final_do_item)[0]

            try:
                numero_do_item = int(valor_do_item)
            except ValueError:
               
                continue

            for i in range(vezes_repetido):
                limite_inferior = i * numero_de_arquivos_por_pasta
                limite_superior = (i + 1) * numero_de_arquivos_por_pasta

                if limite_inferior < numero_do_item <= limite_superior:
                    pasta_raw_estrela_i = os.path.join(pasta_raw_estrela, f"Obs_{i+1}_{NOME_ESTRELA}")

                    if not os.path.exists(pasta_raw_estrela_i):
                        os.makedirs(pasta_raw_estrela_i, exist_ok=True)

                    caminho_destino_completo = os.path.join(pasta_raw_estrela_i, nome_item)
                    shutil.move(caminho_origem_completo, caminho_destino_completo)
                    break

# **Extração de dados**

In [ ]:
#Extrair os arquivos 2D dos fits 3D da pasta calib
arquivos_fits = glob.glob(os.path.join(pasta_raw_calib, '*.fits'))

tem_arquivos_menor_que_3 = False

for arq in arquivos_fits:
    try:
        hdr = fits.getheader(arq)
        naxis = hdr.get('NAXIS', 0)
        
        
        if naxis < 3:
            tem_arquivos_menor_que_3 = True
            break 
            
    except Exception as e:
        print(f"Erro ao ler cabeçalho de {os.path.basename(arq)}: {e}")

if tem_arquivos_menor_que_3:
    print("Pasta Calib já possui ficheiros com dimensão 2D (NAXIS < 3). Descompactação ignorada.")
else:
    print("Todos os ficheiros têm NAXIS 3. A iniciar descompactação...")
    descompactar_cubos_3d(pasta_raw_calib)

In [ ]:
#Extrair os arquivos 2D dos fits 3D da pasta da estrela

for i in range(vezes_repetido):
    pasta_raw_estrela_i = os.path.join(pasta_raw_estrela, f"Obs_{i+1}_{NOME_ESTRELA}")

    if not os.path.exists(pasta_raw_estrela_i):
        continue

    arquivos_fits = glob.glob(os.path.join(pasta_raw_estrela_i, "*.fits"))

    tem_arquivos_2d = False
    for arq in arquivos_fits:
        try:
    
            hdr = fits.getheader(arq)
            if hdr.get('NAXIS') == 2:
                tem_arquivos_2d = True
                break  
        except Exception:
            continue

    if tem_arquivos_2d:
        print(f"Pasta Obs_{i+1}_{NOME_ESTRELA} já possui arquivos com dimensão 2D (NAXIS=2). A descompactação foi ignorada.")
    else:
        descompactar_cubos_3d(pasta_raw_estrela_i)

# **Processamento de Imagens**

In [ ]:
#Quando realmente criamos o MASTER BIAS
padrao_de_busca_em_bias = os.path.join(pasta_reduced_calib,"master_bias*.fits")
arquivos_master_bias_encontrados = glob.glob(padrao_de_busca_em_bias)
if arquivos_master_bias_encontrados:
    print("Master Bias já processados")
else:
    criar_master_bias(pasta_raw_calib, pasta_reduced_calib)

In [ ]:
#Quando realmente criamos o MASTER FLAT

padrao_de_busca_em_flat = os.path.join(pasta_reduced_calib,"master_flat*.fits")
arquivos_master_flat_encontrados = glob.glob(padrao_de_busca_em_flat)

if arquivos_master_flat_encontrados:
    print("Master Flat já processados")
else:
    criar_master_flat(pasta_raw_calib, pasta_reduced_calib)

In [ ]:
#Quando realmente calibramos a imagem

for i in range(vezes_repetido):

    pasta_reduced_estrela_i = f"{pasta_reduced_estrela}/Obs_{i+1}_{NOME_ESTRELA}"
    pasta_raw_estrela_i = f"{pasta_raw_estrela}/Obs_{i+1}_{NOME_ESTRELA}"

    arquivos = os.listdir(pasta_reduced_estrela_i)
    tem_arquivos_na_pasta = any(arquivo.endswith(".fits") for arquivo in arquivos)

    if tem_arquivos_na_pasta:

        print(f"Pasta Obs_{i+1}_{NOME_ESTRELA} já está com arquivos fits,"
              "certifique de que a calibração da imagem não foi feita, mesmo que parcialmente"
        )
    else: 

        reduzir_objetos_paralelo(pasta_raw_estrela_i, pasta_reduced_calib, pasta_reduced_estrela_i)

# **Encontrar o Par e Extração da Fotometria**

In [ ]:
# Parâmetros Fotométricos
raio_abertura = 7.0       # Raio da estrela (px)
r_in_sky = 10.0           # Raio interno do anel de céu (px)
r_out_sky = 15.0          # Raio externo do anel de céu (px)
janela_busca = 12         # Janela max de busca ao redor da coordenada âncora (px)

In [ ]:
# Area das aberturas para subtracao do ceu
area_aper = np.pi * (raio_abertura ** 2)
area_ann = np.pi * (r_out_sky ** 2 - r_in_sky ** 2)

In [ ]:
#Parâmetros das lâminas
mapa_lamina_graus = {
    'L0': 0.0,   'L1': 22.5,  'L2': 45.0,  'L3': 67.5,
    'L4': 90.0,  'L5': 112.5, 'L6': 135.0, 'L7': 157.5,
    'L8': 180.0, 'L9': 202.5, 'LA': 225.0, 'LB': 247.5,
    'LC': 270.0, 'LD': 292.5, 'LE': 315.0, 'LF': 337.5
}

In [ ]:
# Realizar a busca de pares e coleta de dados fotometricos em cada pasta

%matplotlib inline

def preparar_imagem_bgr(data_matrix, usar_log=False):
    data_clean = np.nan_to_num(data_matrix, nan=np.nanmedian(data_matrix), posinf=0, neginf=0)
    
    if usar_log:
        data_clean = np.log1p(np.maximum(data_clean - np.min(data_clean), 0))
        
    vmin = np.median(data_clean)
    vmax = np.percentile(data_clean, 99.5)
    if vmax <= vmin:
        vmax = vmin + 1.0
        
    img_norm = np.clip((data_clean - vmin) / (vmax - vmin) * 255, 0, 255).astype(np.uint8)
    img_flip = cv2.flip(img_norm, 0)
    return cv2.cvtColor(img_flip, cv2.COLOR_GRAY2BGR)


# APLICADOR DE ZOOM E MARGEM

def aplicar_zoom_e_padding(img_draw, zoom_factor):
    h, w = img_draw.shape[:2]
    
    if abs(zoom_factor - 1.0) < 1e-3:
        return img_draw, 0, 0
    elif zoom_factor > 1.0:
        crop_w, crop_h = int(w / zoom_factor), int(h / zoom_factor)
        x1, y1 = (w - crop_w) // 2, (h - crop_h) // 2
        return img_draw[y1:y1+crop_h, x1:x1+crop_w], x1, y1
    else:
        target_w = int(w / zoom_factor)
        target_h = int(h / zoom_factor)
        pad_w = (target_w - w) // 2
        pad_h = (target_h - h) // 2
        
        img_padded = cv2.copyMakeBorder(
            img_draw, pad_h, pad_h, pad_w, pad_w, 
            cv2.BORDER_CONSTANT, value=(0, 0, 0)
        )
        return img_padded, -pad_w, -pad_h


# RECALIBRAÇÃO DE ÂNCORA (COM TODOS OS ATALHOS NA BARRA)

def recalibrar_ancora_cv2(data_frame, x_atual, y_atual, janela_busca):
    h, w = data_frame.shape
    usar_log = False
    zoom_factor = 1.0
    ponto_novo = []
    
    img_bgr = preparar_imagem_bgr(data_frame, usar_log=usar_log)
    nome_janela_base = "RECALIBRACAO DE ANCORA"

    def mouse_cb(event, x_win, y_win, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(ponto_novo) == 0:
            _, offset_x, offset_y = aplicar_zoom_e_padding(img_bgr, zoom_factor)
            
            x_img = offset_x + x_win
            y_img = offset_y + y_win
            
            x_img_clamped = int(np.clip(x_img, 0, w - 1))
            y_img_clamped = int(np.clip(y_img, 0, h - 1))
            y_fits = (h - 1) - y_img_clamped
            
            ponto_novo.append((x_img_clamped, y_fits))

    cv2.namedWindow(nome_janela_base, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(nome_janela_base, 800, 800)
    cv2.setMouseCallback(nome_janela_base, mouse_cb)

    while True:
        img_draw = img_bgr.copy()
        
        yp_cv, xp_cv = int(round((h - 1) - y_atual)), int(round(x_atual))
        cv2.circle(img_draw, (xp_cv, yp_cv), 2, (0, 0, 255), -1)

        if len(ponto_novo) == 0:
            titulo = f"RECALIBRAR ({zoom_factor:.2f}x) -> Clique em P | ATALHOS: [E]Expandir [L]Log [+/Z]ZoomIn [-/X]ZoomOut [0]Reset [ESC]Sair"
        else:
            xn, yn = ponto_novo[0]
            yn_cv, xn_cv = int(round((h - 1) - yn)), int(round(xn))
            cv2.circle(img_draw, (xn_cv, yn_cv), 2, (0, 255, 0), -1)
            cv2.putText(img_draw, "P", (xn_cv + 3, yn_cv - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 255, 0), 1)
            titulo = f"RECALIBRAR ({zoom_factor:.2f}x) -> [ENTER/S]Confirmar [R]Refazer | ATALHOS: [E]Expandir [L]Log [+/Z]ZoomIn [-/X]ZoomOut [ESC]Sair"

        img_view, _, _ = aplicar_zoom_e_padding(img_draw, zoom_factor)

        cv2.setWindowTitle(nome_janela_base, titulo)
        cv2.imshow(nome_janela_base, img_view)
        key = cv2.waitKey(30) & 0xFF

        if key in [13, ord('s'), ord('S')] and len(ponto_novo) > 0:
            break
        elif key in [ord('r'), ord('R')]:
            ponto_novo.clear()
        elif key in [ord('e'), ord('E')]:
            zoom_factor = 0.5 if zoom_factor == 1.0 else 1.0
        elif key in [ord('+'), ord('='), ord('z'), ord('Z')]:
            zoom_factor = min(zoom_factor * 1.3, 8.0)
        elif key in [ord('-'), ord('_'), ord('x'), ord('X')]:
            zoom_factor = max(zoom_factor / 1.3, 0.25)
        elif key == ord('0'):
            zoom_factor = 1.0
        elif key in [ord('l'), ord('L')]:
            usar_log = not usar_log
            img_bgr = preparar_imagem_bgr(data_frame, usar_log=usar_log)
        elif key in [27, ord('q'), ord('Q')]:
            break

    cv2.destroyAllWindows()
    for _ in range(5): cv2.waitKey(1)

    if len(ponto_novo) > 0:
        xn, yn = ponto_novo[0]
        return rastrear_centroide_ancorado(data_frame, xn, yn, janela=janela_busca)
    
    return x_atual, y_atual


# SELEÇÃO DO PAR (COM TODOS OS ATALHOS NA BARRA)

def selecionar_e_confirmar_par_cv2(data_ref, filtro_nome, raio_abertura, r_in_sky, r_out_sky, janela_busca):
    h, w = data_ref.shape
    usar_log = False
    zoom_factor = 1.0
    
    img_bgr = preparar_imagem_bgr(data_ref, usar_log=usar_log)
    
    pontos = []
    x_p_ref, y_p_ref = None, None
    x_s_ref, y_s_ref = None, None
    
    nome_janela = f"SELECAO DE PAR - FILTRO {filtro_nome.upper()}"

    def mouse_callback(event, x_win, y_win, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(pontos) < 2:
            _, offset_x, offset_y = aplicar_zoom_e_padding(img_bgr, zoom_factor)
            
            x_img = offset_x + x_win
            y_img = offset_y + y_win
            
            x_img_clamped = int(np.clip(x_img, 0, w - 1))
            y_img_clamped = int(np.clip(y_img, 0, h - 1))
            y_fits = (h - 1) - y_img_clamped
            
            pontos.append((x_img_clamped, y_fits))

    cv2.namedWindow(nome_janela, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(nome_janela, 800, 800)
    cv2.setMouseCallback(nome_janela, mouse_callback)

    print(f"\n👉 [FILTRO {filtro_nome.upper()}] Clique nas estrelas P e S. Veja os atalhos no topo da janela.\n")

    confirmado = False
    while True:
        img_draw = img_bgr.copy()
        
        if len(pontos) == 0:
            titulo = f"FILTRO {filtro_nome.upper()} ({zoom_factor:.2f}x) -> [1/2] Clique em P | ATALHOS: [E]Expandir [L]Log [+/Z]ZoomIn [-/X]ZoomOut [0]Reset [ESC]Sair"
        elif len(pontos) == 1:
            titulo = f"FILTRO {filtro_nome.upper()} ({zoom_factor:.2f}x) -> [2/2] Clique em S | ATALHOS: [E]Expandir [L]Log [+/Z]ZoomIn [-/X]ZoomOut [0]Reset [ESC]Sair"
            xp, yp_cv = int(pontos[0][0]), int((h - 1) - pontos[0][1])
            cv2.circle(img_draw, (xp, yp_cv), 2, (0, 255, 0), -1)
        elif len(pontos) == 2:
            x_p_ref, y_p_ref = rastrear_centroide_ancorado(data_ref, pontos[0][0], pontos[0][1], janela=janela_busca)
            x_s_ref, y_s_ref = rastrear_centroide_ancorado(data_ref, pontos[1][0], pontos[1][1], janela=janela_busca)
            
            yp_cv, xp_cv = int(round((h - 1) - y_p_ref)), int(round(x_p_ref))
            ys_cv, xs_cv = int(round((h - 1) - y_s_ref)), int(round(x_s_ref))
            
            cv2.circle(img_draw, (xp_cv, yp_cv), int(raio_abertura), (0, 255, 0), 1)
            cv2.circle(img_draw, (xp_cv, yp_cv), int(r_in_sky), (0, 255, 255), 1)
            cv2.circle(img_draw, (xp_cv, yp_cv), int(r_out_sky), (0, 255, 255), 1)
            cv2.putText(img_draw, "P", (xp_cv + 4, yp_cv - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 255, 0), 1)
            
            cv2.circle(img_draw, (xs_cv, ys_cv), int(raio_abertura), (0, 0, 255), 1)
            cv2.circle(img_draw, (xs_cv, ys_cv), int(r_in_sky), (0, 255, 255), 1)
            cv2.circle(img_draw, (xs_cv, ys_cv), int(r_out_sky), (0, 255, 255), 1)
            cv2.putText(img_draw, "S", (xs_cv + 4, ys_cv - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 0, 255), 1)
            
            titulo = f"FILTRO {filtro_nome.upper()} ({zoom_factor:.2f}x) -> [ENTER/S]Confirmar [R]Refazer | ATALHOS: [E]Expandir [L]Log [+/Z]ZoomIn [-/X]ZoomOut [ESC]Pular"

        img_view, _, _ = aplicar_zoom_e_padding(img_draw, zoom_factor)

        cv2.setWindowTitle(nome_janela, titulo)
        cv2.imshow(nome_janela, img_view)
        
        key = cv2.waitKey(30) & 0xFF
        if key in [13, ord('s'), ord('S')] and len(pontos) == 2:
            confirmado = True
            break
        elif key in [ord('r'), ord('R')]:
            pontos.clear()
            print("   🔄 Cliques limpos. Selecione P e S novamente.")
        elif key in [ord('e'), ord('E')]:
            zoom_factor = 0.5 if zoom_factor == 1.0 else 1.0
        elif key in [ord('+'), ord('='), ord('z'), ord('Z')]:
            zoom_factor = min(zoom_factor * 1.3, 8.0)
        elif key in [ord('-'), ord('_'), ord('x'), ord('X')]:
            zoom_factor = max(zoom_factor / 1.3, 0.25)
        elif key == ord('0'):
            zoom_factor = 1.0
        elif key in [ord('l'), ord('L')]:
            usar_log = not usar_log
            img_bgr = preparar_imagem_bgr(data_ref, usar_log=usar_log)
        elif key in [27, ord('q'), ord('Q')]:
            confirmado = False
            break

    cv2.destroyAllWindows()
    for _ in range(5): cv2.waitKey(1)
        
    if confirmado:
        return x_p_ref, y_p_ref, x_s_ref, y_s_ref
    else:
        return None, None, None, None


# LOOP PRINCIPAL DE BUSCA E EXTRAÇÃO FOTOMÉTRICA

for i in range(vezes_repetido):

    pasta_reduced_estrela_i = f"{pasta_reduced_estrela}/Obs_{i+1}_{NOME_ESTRELA}"
        
    todos_arquivos = glob.glob(os.path.join(pasta_reduced_estrela_i, 'red_*.fits'))

    if not todos_arquivos:
        print(f"[AVISO] Nenhum arquivo 'red_*.fits' encontrado em {pasta_reduced_estrela_i}.")
        continue

    arquivos_por_filtro = defaultdict(list)
    for arq in todos_arquivos:
        nome = os.path.basename(arq)
        partes = nome.split('_')
        
        # CORREÇÃO PARA EXTRAIR O FILTRO CORRETAMENTE (MESMO COM 'g5')
        if 'g5' in nome.lower() and len(partes) >= 5:
            filtro = partes[3]  # Pega o filtro na posição 3 ('red_hd148937_g5_v_031.fits')
        elif len(partes) >= 4:
            filtro = partes[2]  # Pega o filtro na posição 2 ('red_48lib_b_001.fits')
        else:
            continue

        arquivos_por_filtro[filtro].append(arq)

    print("=== PIPELINE BLINDADO DE FOTOMETRIA BEACON ===")
    print(f"Filtros identificados na Obs {i+1}: {list(arquivos_por_filtro.keys())}\n")

    for filtro, arquivos in arquivos_por_filtro.items():
        arquivos = sorted(arquivos)
        total_arq = len(arquivos)
        
        print(f"\n{'='*60}")
        print(f"PROCESSANDO FILTRO: '{filtro.upper()}' ({total_arq} imagens)")
        print(f"{'='*60}")
        
        imagem_referencia = arquivos[0]
        data_ref = fits.getdata(imagem_referencia)
        data_ref = np.squeeze(data_ref)
        data_ref = np.nan_to_num(data_ref, nan=np.nanmedian(data_ref), posinf=0, neginf=0)

        # 1. Seleção e Confirmação Direta no OpenCV
        x_prin_ref, y_prin_ref, x_sec_ref, y_sec_ref = selecionar_e_confirmar_par_cv2(
            data_ref, filtro, raio_abertura, r_in_sky, r_out_sky, janela_busca
        )

        if x_prin_ref is None:
            print(f"⚠️ [AVISO] Seleção ignorada/cancelada. Pulando filtro {filtro.upper()}...")
            continue

        dx_rigido = x_sec_ref - x_prin_ref
        dy_rigido = y_sec_ref - y_prin_ref

        # 2. Extração Fotométrica nos Arquivos
        arquivo_saida = os.path.join(pasta_reduced_estrela_i, f'fotometria_filtro_{filtro}.csv')

        with open(arquivo_saida, 'w') as f_out:
            f_out.write("Arquivo,Posicao,Angulo,X_Prin,Y_Prin,Fluxo_Prin,Pico_Prin,X_Sec,Y_Sec,Fluxo_Sec,Pico_Sec,Status\n")
            
            fluxo_p_ref = None
            fluxo_s_ref = None
            ignorar_alertas = False

            for i_arq, arq in enumerate(arquivos):
                nome = os.path.basename(arq)
                try:
                    hdr = fits.getheader(arq)
                    lamina_raw = str(hdr.get('LAMINA', 'L0')).strip().upper()
                    angulo_psi = mapa_lamina_graus.get(lamina_raw, 0.0)
                    posicao_int = int(angulo_psi / 22.5)
                except Exception:
                    posicao_int, angulo_psi = 0, 0.0
                
                dados = fits.getdata(arq)
                dados = np.squeeze(dados)
                dados = np.nan_to_num(dados, nan=np.nanmedian(dados), posinf=0, neginf=0)
                
                x_prin, y_prin = rastrear_centroide_ancorado(dados, x_prin_ref, y_prin_ref, janela=janela_busca)
                x_sec = x_prin + dx_rigido
                y_sec = y_prin + dy_rigido
                
                h, w = dados.shape
                xi_p, yi_p = int(round(x_prin)), int(round(y_prin))
                pico_p = np.max(dados[max(0, yi_p-3):min(h, yi_p+4), max(0, xi_p-3):min(w, xi_p+4)])

                xi_s, yi_s = int(round(x_sec)), int(round(y_sec))
                pico_s = np.max(dados[max(0, yi_s-3):min(h, yi_s+4), max(0, xi_s-3):min(w, xi_s+4)])
                
                aper_p = CircularAperture((x_prin, y_prin), r=raio_abertura)
                ann_p = CircularAnnulus((x_prin, y_prin), r_in=r_in_sky, r_out=r_out_sky)
                phot_p = aperture_photometry(dados, [aper_p, ann_p])
                
                aper_s = CircularAperture((x_sec, y_sec), r=raio_abertura)
                ann_s = CircularAnnulus((x_sec, y_sec), r_in=r_in_sky, r_out=r_out_sky)
                phot_s = aperture_photometry(dados, [aper_s, ann_s])
                
                bkg_p_pixel = phot_p['aperture_sum_1'][0] / area_ann
                fluxo_p = phot_p['aperture_sum_0'][0] - (bkg_p_pixel * area_aper)
                
                bkg_s_pixel = phot_s['aperture_sum_1'][0] / area_ann
                fluxo_s = phot_s['aperture_sum_0'][0] - (bkg_s_pixel * area_aper)
                
                if i_arq == 0:
                    fluxo_p_ref, fluxo_s_ref = max(fluxo_p, 100.0), max(fluxo_s, 100.0)

                # Alertas Inteligentes
                desvio_posicao = np.sqrt((x_prin - x_prin_ref)**2 + (y_prin - y_prin_ref)**2)
                queda_critica = (fluxo_p < 0.20 * fluxo_p_ref) or (fluxo_s < 0.20 * fluxo_s_ref)
                deslocado_critico = desvio_posicao > 8.0

                status_imagem = "VALIDA"
                pular_escrita = False

                if (queda_critica or deslocado_critico) and not ignorar_alertas:
                    print(f"\n⚠️ [ALERTA IMAGEM {i_arq+1}/{total_arq}] {nome}")
                    
                    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
                    vmin, vmax = np.median(dados), np.percentile(dados, 99.5)
                    
                    ax1.imshow(dados, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
                    ax1.add_patch(Circle((x_prin, y_prin), raio_abertura, color='lime', fill=False, lw=2))
                    ax1.set_xlim(x_prin - 25, x_prin + 25)
                    ax1.set_ylim(y_prin - 25, y_prin + 25)
                    ax1.set_title(f"P: Fluxo={fluxo_p:.0f}")
                    
                    ax2.imshow(dados, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
                    ax2.add_patch(Circle((x_sec, y_sec), raio_abertura, color='red', fill=False, lw=2))
                    ax2.set_xlim(x_sec - 25, x_sec + 25)
                    ax2.set_ylim(y_sec - 25, y_sec + 25)
                    ax2.set_title(f"S: Fluxo={fluxo_s:.0f}")
                    
                    plt.tight_layout()
                    plt.show()

                    print("Opções: [ENTER/S] Aceitar | [R] Recalibrar | [M] Mudar Ref | [D] Descartar/Pular Imagem | [C] Desativar Alertas")
                    resp = input("Escolha (ENTER/S/R/M/D/C): ").upper().strip()
                    if resp == 'R':
                        x_prin_ref, y_prin_ref = recalibrar_ancora_cv2(dados, x_prin, y_prin, janela_busca)
                        x_prin, y_prin = x_prin_ref, y_prin_ref
                        x_sec, y_sec = x_prin + dx_rigido, y_prin + dy_rigido
                        print(f"-> Âncora recalibrada para X={x_prin_ref:.1f}, Y={y_prin_ref:.1f}")
                    elif resp == 'M':
                        fluxo_p_ref, fluxo_s_ref = max(fluxo_p, 100.0), max(fluxo_s, 100.0)
                    elif resp == 'D':
                        pular_escrita = True
                        print(f"🚫 Imagem {nome} foi DESCARTADA e não será salva no CSV!")
                    elif resp == 'C':
                        ignorar_alertas = True

                if pular_escrita:
                    continue

                if i_arq % 50 == 0 or i_arq == total_arq - 1:
                    print(f"[{i_arq+1}/{total_arq}] {nome} | Pos {posicao_int} ({angulo_psi}°) -> F_Prin={fluxo_p:.0f} | F_Sec={fluxo_s:.0f}")
                
                f_out.write(f"{nome},{posicao_int},{angulo_psi:.1f},{x_prin},{y_prin},{fluxo_p:.3f},{pico_p:.1f},{x_sec},{y_sec},{fluxo_s:.3f},{pico_s:.1f},{status_imagem}\n") 
                              
        print(f"✅ Filtro {filtro.upper()} finalizado e salvo em {arquivo_saida}!")

print("\n=== EXTRAÇÃO FOTOMÉTRICA CONCLUÍDA COM SUCESSO! ===")

# **Teste de Abertura Fotométrica**

In [ ]:
# TESTE DE ABERTURA FOTOMÉTRICA
obs_teste = 1                              
filtros_teste = None                       
raios_teste = np.arange(4.0, 15.0, 1.0)    

modo_ceu = 'acompanha'   
                         
folga_ceu = 3.0          
largura_ceu = 5.0        


def limites_anel(r):
    if modo_ceu == 'acompanha':
        return r + folga_ceu, r + folga_ceu + largura_ceu
    return r_in_sky, r_out_sky


def fotometria_multi_raio(pasta_red, df, raios):
    """Fluxo (já sem céu) da estrela principal e da secundária para cada raio."""
    n = len(raios)
    fluxo_p = np.full((len(df), n), np.nan)
    fluxo_s = np.full((len(df), n), np.nan)

    for k, (_, lin) in enumerate(df.iterrows()):
        dados = np.squeeze(fits.getdata(os.path.join(pasta_red, lin['Arquivo'])))
        dados = np.nan_to_num(dados, nan=np.nanmedian(dados), posinf=0, neginf=0)

        for (x, y), saida in (((lin['X_Prin'], lin['Y_Prin']), fluxo_p),
                              ((lin['X_Sec'], lin['Y_Sec']), fluxo_s)):
            aberturas = [CircularAperture((x, y), r=r) for r in raios]
            aneis = [CircularAnnulus((x, y), r_in=limites_anel(r)[0], r_out=limites_anel(r)[1]) for r in raios]
            tab = aperture_photometry(dados, aberturas + aneis)

            for j, r in enumerate(raios):
                r_in, r_out = limites_anel(r)
                area_ap = np.pi * r ** 2
                area_an = np.pi * (r_out ** 2 - r_in ** 2)
                ceu_por_pixel = tab[f'aperture_sum_{n + j}'][0] / area_an
                saida[k, j] = tab[f'aperture_sum_{j}'][0] - ceu_por_pixel * area_ap
    return fluxo_p, fluxo_s


def ajustar_polarimetria(psi, fo, fe):
    """Mesma limpeza e mesmo ajuste da célula 'Dados Finais 1'."""
    ok = (fo > 100) & (fe > 100)
    psi, fo, fe = psi[ok], fo[ok], fe[ok]

    zi_raw = (fe - fo) / (fe + fo)
    ok = np.abs(zi_raw) < 0.20
    psi, fo, fe = psi[ok], fo[ok], fe[ok]
    if len(psi) < 8:
        return None

    solver = SLSDualBeamPolarimetry('halfwave', compute_k=True)
    prel = solver.compute(psi, fo, fe)
    residuos = prel.zi.nominal - prel.model(psi * u.deg).nominal
    boa = ~sigma_clip(residuos, sigma=3.0).mask
    res = solver.compute(psi[boa], fo[boa], fe[boa])

    return {
        'N_Usados': int(np.sum(boa)),
        'P_pct': float(res.p.nominal) * 100,
        'P_err': float(res.p.std_dev) * 100,
        'Theta': float(res.theta.nominal),
        'Theta_err': float(res.theta.std_dev),
        'RMS': float(res.rms),
    }


pasta_teste = f"{pasta_reduced_estrela}/Obs_{obs_teste}_{NOME_ESTRELA}"
csvs_teste = sorted(glob.glob(os.path.join(pasta_teste, 'fotometria_filtro_*.csv')))
if not csvs_teste:
    raise FileNotFoundError(f"Nenhum fotometria_filtro_*.csv em {pasta_teste}. Rode a extração fotométrica antes.")

if modo_ceu == 'fixo':
    raios_validos = np.array([r for r in raios_teste if r < r_in_sky])
    if len(raios_validos) < len(raios_teste):
        print(f"[modo 'fixo'] Raios >= r_in_sky ({r_in_sky}) ignorados, o anel de céu pegaria a própria estrela.")
else:
    raios_validos = np.array(raios_teste)

for csv_file in csvs_teste:
    filtro = os.path.basename(csv_file).split('_')[-1].replace('.csv', '')
    if filtros_teste and filtro.lower() not in [f.lower() for f in filtros_teste]:
        continue

    df = pd.read_csv(csv_file)
    df = df[df['Status'] == 'VALIDA'].reset_index(drop=True)
    print(f"\n=== Filtro {filtro.upper()} | {len(df)} imagens | modo de céu: {modo_ceu} ===")

    fluxo_p, fluxo_s = fotometria_multi_raio(pasta_teste, df, raios_validos)
    psi = df['Angulo'].values

    linhas = []
    for j, r in enumerate(raios_validos):
        try:
            res = ajustar_polarimetria(psi, fluxo_p[:, j], fluxo_s[:, j])
        except Exception as e:
            print(f"   raio {r:.1f}: ajuste falhou ({e})")
            continue
        if res is None:
            print(f"   raio {r:.1f}: poucos pontos válidos depois dos filtros")
            continue
        res.update({'Raio': r, 'Fluxo_Medio_P': float(np.nanmean(fluxo_p[:, j]))})
        linhas.append(res)

    if not linhas:
        print("   Nenhum raio gerou resultado.")
        continue

    tabela = pd.DataFrame(linhas)[['Raio', 'N_Usados', 'P_pct', 'P_err', 'Theta', 'Theta_err', 'RMS', 'Fluxo_Medio_P']]
    print(tabela.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    melhor = tabela.loc[tabela['P_err'].idxmin()]
    print(f"-> Menor erro em P: raio = {melhor['Raio']:.1f} px (P = {melhor['P_pct']:.3f} ± {melhor['P_err']:.3f} %)")
    print(f"   Atual no notebook: raio_abertura = {raio_abertura} px")
    print("   Confira se P fica estável entre os raios: se ele muda muito, desconfie de contaminação.")

    tabela.to_csv(os.path.join(pasta_teste, f'teste_abertura_filtro_{filtro}.csv'), index=False)

    fig, eixos = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
    graficos = [('P_err', 'Erro em P (%)'), ('Theta_err', 'Erro em θ (°)'),
                ('RMS', 'RMS dos resíduos'), ('Fluxo_Medio_P', 'Fluxo médio (estrela P)')]
    for ax, (col, rotulo) in zip(eixos.ravel(), graficos):
        ax.plot(tabela['Raio'], tabela[col], 'o-', color='navy')
        ax.axvline(melhor['Raio'], color='green', ls='--', label=f"menor erro P: {melhor['Raio']:.1f}")
        ax.axvline(raio_abertura, color='gray', ls=':', label=f"atual: {raio_abertura}")
        ax.set_ylabel(rotulo)
        ax.grid(True, linestyle=':', alpha=0.6)
    eixos[0, 0].legend(loc='best', fontsize=8)
    eixos[1, 0].set_xlabel('Raio da abertura (px)')
    eixos[1, 1].set_xlabel('Raio da abertura (px)')
    fig.suptitle(f"Teste de abertura - filtro {filtro.upper()} (Obs {obs_teste})")
    plt.tight_layout()
    plt.savefig(os.path.join(pasta_teste, f'teste_abertura_filtro_{filtro}.png'), dpi=200, bbox_inches='tight')
    plt.show()


# **Dados Finais 1**

In [ ]:
# Ajustes para o final
for i in range(vezes_repetido):
    
    pasta_reduced_estrela_i = f"{pasta_reduced_estrela}/Obs_{i+1}_{NOME_ESTRELA}"
    arquivos_csv = sorted(glob.glob(os.path.join(pasta_reduced_estrela_i, 'fotometria_filtro_*.csv')))

    if not arquivos_csv:
        raise FileNotFoundError(f"Nenhum arquivo CSV de fotometria foi encontrado em {pasta_reduced_estrela_i}.")

    # Solvers do astropop (com e sem estimativa dinamica de k)
    pol_solver_k = SLSDualBeamPolarimetry('halfwave', compute_k=True)
    pol_solver_nok = SLSDualBeamPolarimetry('halfwave', compute_k=False)

    resultados_finais = []

    for csv_file in arquivos_csv:
        filtro = os.path.basename(csv_file).split('_')[-1].replace('.csv', '').upper()
        df = pd.read_csv(csv_file)
        
        # Converte forçadamente para numérico
        for col in ['Angulo', 'Fluxo_Prin', 'Fluxo_Sec']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df = df.dropna(subset=['Angulo', 'Fluxo_Prin', 'Fluxo_Sec']).copy()
        
        # 1. Filtra apenas imagens marcadas como VALIDA
        if 'Status' in df.columns:
            df = df[df['Status'] == 'VALIDA'].copy()
        
        # 2. Filtra quadros com fluxos válidos
        df_valido = df[(df['Fluxo_Prin'] > 100) & (df['Fluxo_Sec'] > 100)].copy()
        if len(df_valido) == 0:
            print(f"[{filtro}] Sem dados válidos de fluxo.")
            continue
            
        # 3. Cálculo do Zi bruto e do Fluxo Relativo
        df_valido['Zi_raw'] = (df_valido['Fluxo_Sec'] - df_valido['Fluxo_Prin']) / (df_valido['Fluxo_Sec'] + df_valido['Fluxo_Prin'])
        fluxo_total = df_valido['Fluxo_Prin'] + df_valido['Fluxo_Sec']
        df_valido['Fluxo_rel'] = fluxo_total / np.median(fluxo_total)
        
        # 4. Descarte primário: |Zi| < 0.20 e corte de nuvens ajustado (Fluxo_rel >= 0.85)
        df_limpo = df_valido[
            (np.abs(df_valido['Zi_raw']) < 0.20) & 
            (df_valido['Fluxo_rel'] >= 0.85)
        ].copy()
        
        if len(df_limpo) == 0:
            print(f"[{filtro}] Nenhum dado sobrou após filtros primários.")
            continue

        # Arredonda os ângulos para evitar inconsistências de precisão float
        df_limpo['Angulo'] = np.round(df_limpo['Angulo'].values, 1)
        
        psi = np.asarray(df_limpo['Angulo'].values, dtype=float)
        fo = np.asarray(df_limpo['Fluxo_Prin'].values, dtype=float)
        fe = np.asarray(df_limpo['Fluxo_Sec'].values, dtype=float)
        
        # 5. Ajuste preliminar com fallback inteligente para k
        try:
            res_preliminar = pol_solver_k.compute(psi, fo, fe)
            solver_usado = pol_solver_k
        except Exception as e:
            print(f"⚠️ [{filtro}] Posições incompletas para estimar k ({e}). Usando k=1.0.")
            res_preliminar = pol_solver_nok.compute(psi, fo, fe)
            solver_usado = pol_solver_nok
        
        # 6. Rejeição refinada por 3-Sigma nos resíduos do modelo
        residuos = res_preliminar.zi.nominal - res_preliminar.model(psi * u.deg).nominal
        mascara_boa = ~sigma_clip(residuos, sigma=3.0).mask
        
        # 7. Ajuste Final de Mínimos Quadrados
        try:
            resultado_final = solver_usado.compute(psi[mascara_boa], fo[mascara_boa], fe[mascara_boa])
        except Exception:
            resultado_final = pol_solver_nok.compute(psi[mascara_boa], fo[mascara_boa], fe[mascara_boa])
        
        n_total = len(df)
        n_usados = np.sum(mascara_boa)
        
        p_pct = resultado_final.p.nominal * 100
        p_err = resultado_final.p.std_dev * 100
        theta_deg = resultado_final.theta.nominal
        theta_err = resultado_final.theta.std_dev
        
        k_val = f"{resultado_final.k:.4f}" if hasattr(resultado_final, 'k') and resultado_final.k is not None else "1.0000"
        
        resultados_finais.append({
            'Filtro': filtro,
            'N_Usados': f"{n_usados}/{n_total}",
            'P (%)': f"{p_pct:.3f} ± {p_err:.3f}",
            'Theta (°)': f"{theta_deg:.2f} ± {theta_err:.2f}",
            'k': k_val,
            'RMS': f"{resultado_final.rms:.6f}"
        })
        
        # 8. Plotagem da Curva de Modulação
        plt.figure(figsize=(8, 5))
        
        plt.errorbar(resultado_final.psi.nominal, resultado_final.zi.nominal, 
                    yerr=resultado_final.zi.std_dev, fmt='o', color='crimson', 
                    alpha=0.7, label=f'Medições Válidas (N={n_usados})')
        
        eixo_psi = np.linspace(0, 270, 300)
        modelo_zi = resultado_final.model(eixo_psi * u.deg).nominal
        plt.plot(eixo_psi, modelo_zi, '--', color='navy', lw=2, label=r'Modelo SLS ($Q\cos 4\psi + U\sin 4\psi$)')
        
        plt.title(f"Curva Polarimétrica Limpa - Filtro {filtro}\n$P = {p_pct:.3f}\\% \\pm {p_err:.3f}\\% \\quad \\theta = {theta_deg:.2f}^\\circ \\pm {theta_err:.2f}^\\circ$")
        plt.xlabel(r"Posição da Lâmina Retardadora $\psi$ (graus)")
        plt.ylabel(r"Diferença Relativa de Fluxo $Z_i$")
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.legend(loc='best')
        plt.tight_layout()
        
        nome_grafico = f"curva_polarimetrica_filtro_{filtro}.png"
        caminho_grafico = os.path.join(pasta_reduced_estrela_i, nome_grafico)
        plt.savefig(caminho_grafico, dpi=300, bbox_inches='tight')
        
        plt.show()

    # Salvar o resumo em um arquivo .out
    if resultados_finais:
        print("\n" + "="*70)
        print(f"  RESUMO DA POLARIMETRIA FINAL - {NOME_ESTRELA.upper()} (OBS {i+1})")
        print("="*70)
        
        df_resumo = pd.DataFrame(resultados_finais)
        tabela_texto = df_resumo.to_string(index=False)
        print(tabela_texto)

        caminho_out = os.path.join(pasta_reduced_estrela_i, f"resultados_polarimetria_obs{i+1}_{NOME_ESTRELA}.out")
        
        with open(caminho_out, 'w', encoding='utf-8') as f_out:
            f_out.write("="*70 + "\n")
            f_out.write(f"  RESUMO DA POLARIMETRIA FINAL - {NOME_ESTRELA.upper()} (OBS {i+1})\n")
            f_out.write("="*70 + "\n\n")
            f_out.write(tabela_texto + "\n")
            
        print(f"\n✅ Resultados numéricos salvos em: {caminho_out}\n")